<a href="https://colab.research.google.com/github/DinRazar/ispm/blob/main/%D0%A0%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Задание №7. Проектирование базы данных для информационной системы**

## **Введение**

База данных является фундаментом любой информационной системы, обеспечивая надежное хранение, эффективный доступ и целостность данных. В контексте управления проектами создания информационных систем грамотное проектирование базы данных критически важно для успеха всего проекта, поскольку ошибки на этом этапе могут привести к серьезным проблемам производительности, масштабируемости и поддержки системы в будущем.

## **Формулировка задания**

На основе ранее разработанных концептуальных основ проекта, иерархической структуры работ, спецификации требований к системе (SRS) и UML-диаграмм, спроектируйте реляционную базу данных для вашей информационной системы.

Результатом работы должна стать ER-диаграмма (Entity-Relationship Diagram), отражающая структуру данных, необходимых для функционирования вашего проекта.

# Проектирование реляционной базы данных для MaRS Land Cover Platform

## 1. Назначение ER-диаграммы

ER-диаграмма (Entity-Relationship Diagram) предназначена для описания структуры данных информационной системы на концептуальном и логическом уровне.  
Она показывает:

- какие сущности (таблицы) должны храниться в базе данных;
- какие атрибуты необходимы для каждой сущности;
- какие связи существуют между сущностями;
- какие ключи и ограничения обеспечивают целостность данных.

Для проекта **MaRS Land Cover Platform** ER-диаграмма строится на основе ранее выполненных этапов проектирования:  
концепции системы мониторинга земных покровов, WBS, SRS и UML-диаграмм, в которых уже были выделены пользователи, запросы анализа, сцены спутниковых данных, результаты классификации и метрики качества.

---

## 2. Цель проектирования базы данных

Цель проектирования базы данных состоит в создании структуры, которая позволит:

- хранить учётные записи пользователей и их роли;
- сохранять параметры запросов анализа (область интереса, временной интервал, тип анализа, модель);
- учитывать используемые спутниковые сцены из внешних источников;
- хранить результаты классификации земных покровов;
- сохранять статистику по классам и метрики качества;
- обеспечивать связь между пользовательскими действиями, входными данными и результатами вычислений.

Такая структура необходима для корректной работы веб-интерфейса, REST API, аналитического модуля и слоя хранения данных, что напрямую соответствует функциональным и нефункциональным требованиям SRS.

---

## 3. Логика выбора сущностей

На основе SRS и UML-диаграмм в реляционной модели целесообразно выделить следующие основные сущности:

1. **roles** – роли пользователей;
2. **users** – пользователи системы;
3. **analysis_requests** – запросы на анализ;
4. **data_sources** – источники спутниковых данных;
5. **satellite_scenes** – загруженные спутниковые сцены;
6. **request_scenes** – связь между запросом анализа и сценами;
7. **classification_results** – результаты классификации;
8. **quality_metrics** – метрики качества результата;
9. **land_cover_classes** – справочник классов земных покровов;
10. **result_class_stats** – статистика по классам для конкретного результата.

Такой состав сущностей отражает и пользовательский сценарий системы, и внутреннюю обработку данных:  
пользователь создаёт запрос анализа, система подбирает и загружает сцены из определённого источника, затем формирует результат классификации, рассчитывает метрики качества и сохраняет статистику по классам земных покровов.
---

## 4. Описание сущностей и атрибутов

### 4.1. Таблица `roles`

Назначение: хранение ролей пользователей.

**Основные поля:**

- `role_id` – первичный ключ;
- `role_name` – название роли (`ADMIN`, `ANALYST`, `VIEWER`);
- `description` – описание роли.

Роль используется для разграничения прав доступа в системе администрирования и выполнения аналитических сценариев.

---

### 4.2. Таблица `users`

Назначение: хранение учётных записей пользователей системы.

**Основные поля:**

- `user_id` – первичный ключ;
- `username` – уникальное имя пользователя;
- `email` – адрес электронной почты;
- `password_hash` – хэш пароля;
- `role_id` – внешний ключ на таблицу `roles`;
- `created_at` – дата создания учётной записи;
- `is_active` – признак активности пользователя.

Один пользователь принадлежит одной роли, но одна роль может быть назначена нескольким пользователям.

---

### 4.3. Таблица `analysis_requests`

Назначение: хранение запросов на запуск анализа и классификации.

**Основные поля:**

- `request_id` – первичный ключ;
- `user_id` – внешний ключ на пользователя, создавшего запрос;
- `aoi_geom` – геометрия области интереса (Polygon / MultiPolygon);
- `date_from` – начало временного интервала;
- `date_to` – конец временного интервала;
- `analysis_type` – тип анализа (`optical`, `sar`, `multimodal`);
- `model_name` – используемая модель (`MaRS`, `RNN`, `LSTM`);
- `status` – статус выполнения (`PENDING`, `RUNNING`, `DONE`, `FAILED`);
- `created_at` – дата создания запроса.

Эта таблица является центральной для всей системы, так как именно от запроса анализа строится дальнейшая цепочка обработки спутниковых данных.

---

### 4.4. Таблица `data_sources`

Назначение: хранение справочника внешних источников данных.

**Основные поля:**

- `source_id` – первичный ключ;
- `source_name` – название источника (`Copernicus`, `USGS`, `GEE`);
- `api_url` – базовый URL API;
- `description` – краткое описание;
- `is_active` – признак доступности/использования.

Таблица позволяет хранить и расширять набор интеграций с внешними провайдерами спутниковых данных, упомянутыми в SRS и анализе аналогов.

---

### 4.5. Таблица `satellite_scenes`

Назначение: хранение сведений о найденных и загруженных спутниковых сценах.

**Основные поля:**

- `scene_id` – первичный ключ;
- `source_id` – внешний ключ на `data_sources`;
- `external_scene_code` – идентификатор сцены во внешнем каталоге;
- `platform` – платформа (`Sentinel-1`, `Sentinel-2`, `Landsat`);
- `acquisition_date` – дата и время съёмки;
- `cloud_coverage` – облачность (для оптических сцен);
- `footprint_geom` – геометрия покрытия сцены;
- `download_url` – ссылка на загрузку;
- `local_path` – путь к локально сохранённому файлу;
- `created_at` – дата регистрации сцены в системе.

Одна сцена принадлежит одному источнику данных, а одна запись источника может быть связана со многими сценами.

---

### 4.6. Таблица `request_scenes`

Назначение: реализация связи многие-ко-многим между запросами анализа и спутниковыми сценами.

**Основные поля:**

- `request_id` – внешний ключ на `analysis_requests`;
- `scene_id` – внешний ключ на `satellite_scenes`.

**Составной первичный ключ:**

- (`request_id`, `scene_id`)

Один запрос анализа может использовать несколько сцен, и одна сцена может участвовать в нескольких аналитических запросах.

---

### 4.7. Таблица `classification_results`

Назначение: хранение результатов классификации по каждому запросу.

**Основные поля:**

- `result_id` – первичный ключ;
- `request_id` – внешний ключ на `analysis_requests`;
- `raster_path` – путь к GeoTIFF с результатом;
- `preview_path` – путь к изображению предпросмотра или GIF/MP4;
- `created_at` – дата создания результата;
- `completed_at` – дата завершения расчёта.

Для одного запроса обычно формируется один основной результат классификации, поэтому связь между `analysis_requests` и `classification_results` можно трактовать как 1:1 или 1:0..1.

---

### 4.8. Таблица `quality_metrics`

Назначение: хранение метрик качества классификации.

**Основные поля:**

- `metric_id` – первичный ключ;
- `result_id` – внешний ключ на `classification_results`;
- `overall_accuracy` – общая точность (OA);
- `kappa` – коэффициент каппа;
- `mean_iou` – среднее значение IoU;
- `reference_source` – источник эталонных данных (`Dynamic World`, `ESA WorldCover`);
- `calculated_at` – дата расчёта.

Эта сущность отражает требования к оценке качества моделей по Accuracy, IoU и другим метрикам, упомянутым в WBS и SRS.

---

### 4.9. Таблица `land_cover_classes`

Назначение: хранение справочника классов земных покровов.

**Основные поля:**

- `class_id` – первичный ключ;
- `class_code` – код класса;
- `class_name` – название класса (`water`, `vegetation`, `bare_ground`, `burn`, `snow` и т.д.);
- `color_hex` – цвет для визуализации класса на карте;
- `description` – описание класса.

Эта таблица нужна для единообразного хранения классификационных категорий и их визуального представления.

---

### 4.10. Таблица `result_class_stats`

Назначение: хранение статистики по каждому классу в составе конкретного результата.

**Основные поля:**

- `result_id` – внешний ключ на `classification_results`;
- `class_id` – внешний ключ на `land_cover_classes`;
- `area_ha` – площадь класса в гектарах;
- `percentage` – доля класса в процентах.

**Составной первичный ключ:**

- (`result_id`, `class_id`)

Таблица позволяет хранить детализированную статистику по классам земных покровов для каждого результата анализа.

---

## 5. Связи между сущностями

Основные связи в ER-модели:

- **roles (1) — (M) users**  
  Одна роль может быть назначена нескольким пользователям.

- **users (1) — (M) analysis_requests**  
  Один пользователь может создавать много запросов анализа.

- **data_sources (1) — (M) satellite_scenes**  
  Один источник данных содержит множество сцен.

- **analysis_requests (M) — (M) satellite_scenes**  
  Реализуется через промежуточную таблицу `request_scenes`.

- **analysis_requests (1) — (1) classification_results**  
  Один запрос формирует один основной результат классификации.

- **classification_results (1) — (1) quality_metrics**  
  Для каждого результата сохраняется один набор итоговых метрик качества.

- **classification_results (1) — (M) result_class_stats**  
  Один результат содержит статистику по нескольким классам.

- **land_cover_classes (1) — (M) result_class_stats**  
  Один класс может встречаться во многих результатах.

Такая структура обеспечивает нормализованное хранение данных и исключает избыточное дублирование.

---

## 6. Обоснование выбранной структуры

Предложенная реляционная модель отражает архитектуру системы, ранее определённую в UML-диаграммах, и поддерживает все ключевые пользовательские и системные сценарии.  
Сущность `analysis_requests` играет роль центральной бизнес-сущности, так как именно она связывает пользователя, входные параметры анализа, используемые спутниковые сцены и итоговые результаты. Это соответствует логике SRS, где запуск классификации строится вокруг области интереса, временного интервала, выбранной модели и дальнейшего формирования результата.
Выделение отдельных таблиц для сцен, источников данных, метрик качества и статистики по классам позволяет избежать избыточности и делает модель расширяемой. Например, можно без изменения общей логики добавить новый внешний источник данных, новую ML‑модель или новый класс земных покровов. Кроме того, такая схема подходит как для веб‑приложения, так и для REST API, так как позволяет легко получать как агрегированные данные (запрос, статус, результат), так и детальные сведения (использованные сцены, метрики, распределение по классам).

Использование отдельного справочника ролей и классов земных покровов повышает управляемость системы и упрощает сопровождение. Пространственные атрибуты (`aoi_geom`, `footprint_geom`) логично хранить в PostgreSQL/PostGIS, что соответствует ранее выбранному стеку проекта и требованиям к работе с геоданными.

---

## 7. Текстовое представление ER-диаграммы

Ниже приведено логическое текстовое представление связей ER-диаграммы:

- `roles` 1 ---- M `users`
- `users` 1 ---- M `analysis_requests`
- `data_sources` 1 ---- M `satellite_scenes`
- `analysis_requests` M ---- M `satellite_scenes` (через `request_scenes`)
- `analysis_requests` 1 ---- 1 `classification_results`
- `classification_results` 1 ---- 1 `quality_metrics`
- `classification_results` 1 ---- M `result_class_stats`
- `land_cover_classes` 1 ---- M `result_class_stats`

---



## 8. Сама ER-диаграмма

ur.svg

## 9. Вывод

Разработанная ER-диаграмма отражает структуру данных, необходимую для функционирования системы мониторинга и классификации земных покровов MaRS Land Cover Platform.  
Она согласована с ранее разработанными концептуальными материалами, WBS, SRS и UML-диаграммами, а также поддерживает ключевые бизнес-процессы системы: управление пользователями, формирование запросов анализа, обработку спутниковых данных, хранение результатов и оценку качества.

Предложенная модель является нормализованной, расширяемой и пригодной для реализации в PostgreSQL/PostGIS как основной реляционной СУБД проекта.